In [64]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pandas_datareader.famafrench import get_available_datasets
import pandas_datareader.data as web

In [4]:
#plot 한글 적용
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

### 파마 프렌치 패터 자료 다운로드
#### https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

In [32]:
def getFmaFranchFactor(START_DATE, END_DATE):
    
    # 'Annual Factors: January-December' 문구를 기준으로 웓단위, 년단위 데이터가 함께 들어있는 자료
    factor_df = pd.read_csv('F-F_Research_Data_Factors.csv', skiprows=3)
    
    STR_TO_MATCH = ' Annual Factors: January-December '
    indices = factor_df.iloc[:, 0] == STR_TO_MATCH
    
    start_of_annual = factor_df[indices].index[0] 
    
    factor_df = factor_df[factor_df.index < start_of_annual]
    
    # mkt(시장위험 프리미엄), smb(소형주 − 대형주), hml(가치주 − 성장주), rf(무위험 수익률)
    factor_df.columns = ['date', 'mkt', 'smb', 'hml', 'rf']  

    # dateTime형으로 변환
    factor_df['date'] = pd.to_datetime(factor_df['date'], format='%Y%m')
    
    # 날짜를 인덱스로 설정
    factor_df = factor_df.set_index('date').to_period('M')

    #필요한 기간만 추출
    factor_df = factor_df.loc[START_DATE:END_DATE]

    return factor_df

### 데이터 불러오기(META 2014~2018년도)

In [54]:
file_path = "../Data-collection/dailyStock"
START_DATE = '2014-01-01'
END_DATE = '2018-12-31'
target = "META"

file_path = f"{file_path}/{target}.csv"
meta_df = pd.read_csv(file_path, encoding="utf-8")

meta_df['Date'] = pd.to_datetime(meta_df['Date']) # 날짜로 변환
meta_df = meta_df.set_index('Date')               # Date를 인덱스로 설정
meta_df = meta_df.loc[START_DATE:END_DATE]  # 분석 기간

y= meta_df['adj_close'].resample("ME").last().pct_change().dropna()

y.index = y.index.to_period('M')
y.name = 'rtn'

In [55]:
factor_df = getFmaFranchFactor(START_DATE, END_DATE)
##백분율 데이터를 소수로 변경
factor_df = factor_df.apply(pd.to_numeric, errors='coerce').div(100)

In [56]:
## join 후 초과 수익률 계산
ff_data = factor_df.join(y)
ff_data['excess_rtn'] = ff_data.rtn - ff_data.rf

In [57]:
#종속변수 : 초과수익률(excess_rtn) 
#독립변수 : smb(규모요인 +:소형주 -:대형주), hml(가치요인 +:가치주 -:성장주), rf(무위험 수익률)
ff_model = smf.ols(formula='excess_rtn ~ mkt + smb + hml',  
                   data=ff_data).fit()

ㅁ
print(ff_model.summary())

                            OLS Regression Results                            
Dep. Variable:             excess_rtn   R-squared:                       0.247
Model:                            OLS   Adj. R-squared:                  0.206
Method:                 Least Squares   F-statistic:                     6.023
Date:                Fri, 18 Sep 2026   Prob (F-statistic):            0.00127
Time:                        16:27:49   Log-Likelihood:                 89.458
No. Observations:                  59   AIC:                            -170.9
Df Residuals:                      55   BIC:                            -162.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0071      0.007      0.950      0.3